In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms

import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.train import net_train_AnyNet_L, net_train_ViT_L, net_train_RNN_L, net_train_LC_L
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code
Library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
################################################################################
import subprocess
import sys
required = {'ONE-api', 'brain', 'ibllib'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *required])

from one.api import ONE
from brainbox.io.one import SessionLoader, SpikeSortingLoader
# from ibllib.atlas import AllenAtlas
# from brainbox.io.spikeglx import Streamer
# from neurodsp.voltage import destripe
# from datetime import datetime
# from pprint import pprint

# ba = AllenAtlas()
# br = ba.regions
# ba.compute_regions_volume()


In [3]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-e1e8ffe1-0827-d7c1-1c99-90e8dc9ecf51)


In [4]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [5]:
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    # 'epochs':50,
    # 'save_dir':'/content/drive/MyDrive/Project/BrainRegionId/Project43',
}


In [6]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project43/Result/brain_signal_lfp/brain_signal_lfp.pt', weights_only=False)
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)

brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']
acronym_selec_list = list_dict['acronym_selec_list']

In [7]:
list_dict.keys()

dict_keys(['brain_region_index', 'brain_region_index_Cosmos', 'brain_region_atlas', 'subject_list', 'exp_list', 'key_list', 'coordinate_list', 'depth_list', 'volume_list', 'brain_signal_lfp', 'brain_signal_ap', 'train_list_key', 'test_list_key', 'train_list_subject', 'test_list_subject', 'train_list_exp', 'test_list_exp', 'train_list_trial', 'test_list_trial', 'train_list_intest', 'test_list_intest', 'acronym_selec_list', 'valid_list_intest'])

In [8]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [ ]:
device.type

'cuda'

In [13]:
ind = 0

key = 'stimOff_times'

model_dir = '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static/Model'

if device.type != 'cuda':

    Classifier_LCC = torch.load(model_dir + f'/LC_L_Allen_chance_{key}_{ind}.pth', map_location=torch.device('cpu'))
    Classifier_LC = torch.load(model_dir + f'/LC_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))


    Classifier_AnyNet = torch.load(model_dir + f'/AnyNet_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))
    Classifier_ViT = torch.load(model_dir + f'/ViT_L_Allen_{key}_final_{ind}.pth', map_location=torch.device('cpu'))
    Classifier_RNN = torch.load(model_dir + f'/RNN_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))

elif device.type == 'cuda':

    Classifier_LCC = torch.load(model_dir + f'/LC_L_Allen_chance_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_LC = torch.load(model_dir + f'/LC_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)


    Classifier_AnyNet = torch.load(model_dir + f'/AnyNet_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_ViT = torch.load(model_dir + f'/ViT_L_Allen_{key}_final_{ind}.pth', weights_only=False).to(device)
    Classifier_RNN = torch.load(model_dir + f'/RNN_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)




In [14]:
subject_od_ind = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen' + f'/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
subject_od_list = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen' + f'/subject_od_list_Allen_{key}{0}.pt', weights_only=False)

In [15]:
train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index[test_ind], coordinate_list[test_ind], acronym_selec_list[test_ind])

test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

In [16]:
Classifier_name = ['AnyNet', 'ViT', 'RNN']
Classifier_list = [Classifier_AnyNet, Classifier_ViT, Classifier_RNN]
acu_test = []
acronym_test = []
acronym_index_test = []
model_name = []
subject_num = []
for Classifier_ii, Classifier in enumerate(Classifier_list):
    for acronym_ii, acronym in enumerate(acronym_list):

        test_ind_acronym_ii = np.intersect1d(np.argwhere(np.array(list_dict['brain_region_atlas']) == acronym).flatten(), test_ind)

        if len(test_ind_acronym_ii) < 1:

            acronym_test.append(acronym)

            acronym_index_test.append(acronym_ii)

            acu_test.append(np.nan)

            model_name.append(Classifier_name[Classifier_ii])
        else:
            subject_num.append(np.unique(np.array(list_dict['subject_list'])[test_ind_acronym_ii]))
            test_indiv = test_ind_acronym_ii
            data_test_indiv = TensorDataset(brain_signal_lfp[test_indiv,:], brain_region_index[test_indiv])
            test_iter_indiv = DataLoader(data_test_indiv, batch_size=128, shuffle=True)

            for x_test1, y_test in test_iter_indiv:

                x_test = lfp_spectro(x_test1, spectro_args, train_args)
                y_test = y_test.to(device)


                if Classifier_name[Classifier_ii] == 'RNN':
                    py_test = Classifier(x_test.to(device).squeeze(1).permute(0, 2, 1))
                elif Classifier_name[Classifier_ii] in ['Chance', 'Linear']:
                    py_test = Classifier(x_test.to(device).squeeze(1).flatten(start_dim=1))
                else:
                    py_test = Classifier(x_test.to(device))

                # print(acronym, f'acu: {(torch.sum(torch.argmax(py_test, dim=1) == y_test) / y_test.size(0)).detach().cpu()}')
                acronym_test.append(acronym)

                acronym_index_test.append(acronym_ii)

                acu_test.append(accu_fun(py_test, y_test))

                model_name.append(Classifier_name[Classifier_ii])

        print(f'Classifier{Classifier_ii}: acronym ii: {acronym_ii}', )


Classifier0: acronym ii: 0
Classifier0: acronym ii: 1
Classifier0: acronym ii: 2
Classifier0: acronym ii: 3
Classifier0: acronym ii: 4
Classifier0: acronym ii: 5
Classifier0: acronym ii: 6
Classifier0: acronym ii: 7
Classifier0: acronym ii: 8
Classifier0: acronym ii: 9
Classifier0: acronym ii: 10
Classifier0: acronym ii: 11
Classifier0: acronym ii: 12
Classifier0: acronym ii: 13
Classifier0: acronym ii: 14
Classifier0: acronym ii: 15
Classifier0: acronym ii: 16
Classifier0: acronym ii: 17
Classifier0: acronym ii: 18
Classifier0: acronym ii: 19
Classifier0: acronym ii: 20
Classifier0: acronym ii: 21
Classifier0: acronym ii: 22
Classifier0: acronym ii: 23
Classifier0: acronym ii: 24
Classifier0: acronym ii: 25
Classifier0: acronym ii: 26
Classifier0: acronym ii: 27
Classifier0: acronym ii: 28
Classifier0: acronym ii: 29
Classifier0: acronym ii: 30
Classifier0: acronym ii: 31
Classifier0: acronym ii: 32
Classifier0: acronym ii: 33
Classifier0: acronym ii: 34
Classifier0: acronym ii: 35
Cl

In [17]:
acu_test_dict = pd.DataFrame({
    'acu_test': np.array(acu_test),
    'name': model_name,
    'acronym_test': acronym_test,
    'acronym_index_test': acronym_index_test,
})


In [18]:
torch.save(acu_test_dict, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static/results/acronym_test_dict.pt')
torch.save(subject_num, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static/results/subject_stat.pt')

In [19]:
from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
br = ba.regions
ba.compute_regions_volume()


Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/average_template_25.nrrd Bytes: 32998960


100%|██████████| 31.470260620117188/31.470260620117188 [00:02<00:00, 14.55it/s]


Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/annotation_25.nrrd Bytes: 4035363


100%|██████████| 3.848422050476074/3.848422050476074 [00:00<00:00, 39.98it/s]


In [20]:
acronym_id_marked = []
acu_id_marked = []
acronym_marked = []
for acronym_ii, acronym in enumerate(acronym_list):
    acu_test_dict_selec = acu_test_dict[acu_test_dict['acronym_test'] == acronym]
    if len(acu_test_dict_selec) < 1:
        continue
    mean_max = 0
    for name0 in Classifier_name:
        # print(acu_test_dict_selec[acu_test_dict_selec['name'] == name0]['acu_test'])
        mean0 = acu_test_dict_selec[acu_test_dict_selec['name'] == name0]['acu_test'].mean()
        # print(mean0)
        if mean0 > mean_max:
            mean_max = mean0
    if (mean_max > 0.75) or (acronym in select_acronym_list):
        print(acronym_ii)
        print(acronym)
        acronym_id_marked.append(acronym_ii)
        acu_id_marked.append(mean_max)
        acronym_marked.append(acronym)

NameError: name 'select_acronym_list' is not defined

In [21]:
sns.color_palette('colorblind')

[(0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
 (0.8705882352941177, 0.5607843137254902, 0.0196078431372549),
 (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
 (0.8352941176470589, 0.3686274509803922, 0.0),
 (0.8, 0.47058823529411764, 0.7372549019607844),
 (0.792156862745098, 0.5686274509803921, 0.3803921568627451),
 (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
 (0.5803921568627451, 0.5803921568627451, 0.5803921568627451),
 (0.9254901960784314, 0.8823529411764706, 0.2),
 (0.33725490196078434, 0.7058823529411765, 0.9137254901960784)]

In [22]:
Classifier_name = ['AnyNet', 'ViT', 'RNN']
Classifier_list = [Classifier_AnyNet, Classifier_ViT, Classifier_RNN]
subject_n = []
acronym_stat = []
for Classifier_ii, Classifier in enumerate(Classifier_list):
    for acronym_ii, acronym in enumerate(acronym_list):

        test_ind_acronym_ii = np.intersect1d(np.argwhere(np.array(list_dict['brain_region_atlas']) == acronym).flatten(), test_ind)

        if len(test_ind_acronym_ii) < 1:
            continue

        else:
            subject_n.append(len(np.unique(np.array(list_dict['subject_list'])[test_ind_acronym_ii])))
            acronym_stat.append(acronym_ii)
            print(acronym_ii)

    break

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
68
69
73
76
77
78
80
81
82
83
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
201
202
203
204
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
282
283
284
285
286
287


In [23]:
torch.save(acronym_stat, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static/results/acronym_stat.pt')
torch.save(subject_n, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static/results/subject_n.pt')

In [24]:
model_type = []
acu_test_overall = []
Classifier_name = ['Chance',  'Linear', 'AnyNet', 'ViT', 'RNN']
for Classifier_ii, Classifier in enumerate([Classifier_LCC, Classifier_LC, Classifier_AnyNet, Classifier_ViT, Classifier_RNN]):
    Classifier.eval()
    for x_test1, y_test, coordinate_test, acronym_selec_test in test_iter:
        if Classifier_name[Classifier_ii] == 'RNN':
            x_test = lfp_spectro(x_test1, spectro_args, train_args)
            y_test = y_test.to(device)
            py_test = Classifier(x_test.to(device).squeeze(1).permute(0, 2, 1))
            del x_test, x_test1
            acu_test_overall.append(accu_fun(py_test, y_test))
            model_type.append(Classifier_name[Classifier_ii])

        elif Classifier_name[Classifier_ii] in ['Chance', 'Linear']:
            x_test = lfp_spectro(x_test1, spectro_args, train_args)
            y_test = y_test.to(device)
            py_test = Classifier(x_test.to(device).squeeze(1).flatten(start_dim=1))
            del x_test, x_test1
            acu_test_overall.append(accu_fun(py_test, y_test))
            model_type.append(Classifier_name[Classifier_ii])

        else:
            x_test = lfp_spectro(x_test1, spectro_args, train_args)
            y_test = y_test.to(device)
            py_test = Classifier(x_test.to(device))
            del x_test, x_test1
            acu_test_overall.append(accu_fun(py_test, y_test))
            model_type.append(Classifier_name[Classifier_ii])
    print(Classifier_ii)


acu_overall = pd.DataFrame({
    'acu_test_overall': np.array(acu_test_overall),
    'model_type': model_type,
})

0
1
2
3
4


In [25]:
torch.save(acu_overall, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/static/results/acu_overall.pt')

In [ ]:
from google.colab import runtime
runtime.unassign()